Comprobar si tengo alguna columna en el obs que indique el tipo de tumor o el origen de cada célula

In [11]:
import scanpy as sc
adata = sc.read_h5ad("/beegfs/home/iruizdealda/HCC_singlecell_project/data/adata_GSE125449_raw.h5ad")
print(adata.obs.columns.tolist())
print(adata.obs.head())

['batch']
                     batch
AAACCTGAGGCGTACA-1-0     0
AAACGGGAGATCGATA-1-0     0
AAAGCAAAGATCGGGT-1-0     0
AAATGCCGTCTCAACA-1-0     0
AACACGTCACGGCTAC-1-0     0


<div style="text-align: justify">
El dataset GSE125449 contiene células tanto de HCC como de iCCA. 
Dado que el objetivo del estudio es el análisis de HCC, se filtraron 
las células de iCCA antes de continuar con el pipeline. La identificación 
del tipo tumoral se realizó a partir del archivo de metadata original 
del paper (GSE125449_SetX_samples.txt), donde el prefijo del Sample ID 
indica el subtipo histológico: H para HCC y C para iCCA (colangiocarcinoma 
intrahepático), tal como se describe en Ma et al., 2019.
</div>


In [12]:
import pandas as pd

adata = sc.read_h5ad("/beegfs/home/iruizdealda/HCC_singlecell_project/data/adata_GSE125449_raw.h5ad")

# Leer los samples de Set1 y Set2
s1 = pd.read_csv("/beegfs/home/iruizdealda/HCC_singlecell_project/data/GSE125449/Set1/GSE125449_Set1_samples.txt", sep="\t")
s2 = pd.read_csv("/beegfs/home/iruizdealda/HCC_singlecell_project/data/GSE125449/Set2/GSE125449_Set2_samples.txt", sep="\t")

samples = pd.concat([s1, s2])
print(samples.head())
print(samples.columns.tolist())

          Sample        Cell Barcode Type
0  S02_P01_LCP21  AAACCTGAGGCGTACA-1  CAF
1  S02_P01_LCP21  AAACGGGAGATCGATA-1  CAF
2  S02_P01_LCP21  AAAGCAAAGATCGGGT-1  CAF
3  S02_P01_LCP21  AAATGCCGTCTCAACA-1  CAF
4  S02_P01_LCP21  AACACGTCACGGCTAC-1  TEC
['Sample', 'Cell Barcode', 'Type']


Ver qué samples únicos hay en Set1 y Set2 para identificar cuáles son H (HCC) y cuáles C (iCCA)

In [13]:
print("Samples únicos Set1 y Set2:")
print(samples["Sample"].unique())

Samples únicos Set1 y Set2:
['S02_P01_LCP21' 'S07_P02_LCP28' 'S08_P03_LCP26' 'S09_P04_LCP25'
 'S10_P05_LCP23' 'S11_P06_LCP29' 'S12_P07_LCP30' 'S15_P09_LCP38'
 'S16_P10_LCP18' 'S19_P11_LCP39' 'S20_P12_LCP35' 'S21_P13_LCP37'
 'S300_P02_LCP60' 'S305_P06_LCP56' 'S351_P10_LCP34' 'S355_P13_LCP42'
 'S358_P16_LCP46' 'S364_P21_LCP65' 'S365_P22_LCP66']


Los nombres de sample no tienen el prefijo H/C directamente, sino que usan el código LCP. Hay que cruzarlo con la Tabla 1 del paper que ya tenemos. Mirando la tabla:

HCC → LCP21, LCP28, LCP23, LCP30, LCP18, LCP37, LCP34, LCP65

iCCA → LCP26, LCP25, LCP29, LCP38, LCP39, LCP35, LCP60, LCP56, LCP42, LCP46, LCP66

In [14]:
# Mapeo LCP → HCC/iCCA según Tabla 1 del paper (Ma et al. 2019)
hcc_samples = ["LCP21", "LCP28", "LCP23", "LCP30", "LCP18", "LCP37", "LCP34", "LCP65"]

# Identificar qué samples son HCC basándonos en el sufijo LCP
samples["tumor_type"] = samples["Sample"].apply(
    lambda x: "HCC" if any(lcp in x for lcp in hcc_samples) else "iCCA"
)

print(samples["tumor_type"].value_counts())
print("\nVerificación:")
print(samples.groupby(["Sample", "tumor_type"]).size().reset_index(name="n_cells"))

tumor_type
iCCA    7079
HCC     2867
Name: count, dtype: int64

Verificación:
            Sample tumor_type  n_cells
0    S02_P01_LCP21        HCC      704
1    S07_P02_LCP28        HCC      124
2    S08_P03_LCP26       iCCA      299
3    S09_P04_LCP25       iCCA      207
4    S10_P05_LCP23        HCC      151
5    S11_P06_LCP29       iCCA      939
6    S12_P07_LCP30        HCC      805
7    S15_P09_LCP38       iCCA     1046
8    S16_P10_LCP18        HCC      124
9    S19_P11_LCP39       iCCA      445
10   S20_P12_LCP35       iCCA      139
11   S21_P13_LCP37        HCC      132
12  S300_P02_LCP60       iCCA     1418
13  S305_P06_LCP56       iCCA      137
14  S351_P10_LCP34        HCC      238
15  S355_P13_LCP42       iCCA      508
16  S358_P16_LCP46       iCCA      585
17  S364_P21_LCP65        HCC      589
18  S365_P22_LCP66       iCCA     1356


Filtramos

In [15]:
# Barcodes de células HCC
hcc_barcodes = samples[samples["tumor_type"] == "HCC"]["Cell Barcode"].tolist()
print(f"Barcodes HCC en metadata: {len(hcc_barcodes)}")

# Ver cómo son los barcodes en el AnnData para hacer el match
print("\nEjemplo barcodes AnnData:")
print(adata.obs_names[:5].tolist())

print("\nEjemplo barcodes metadata HCC:")
print(hcc_barcodes[:5])

Barcodes HCC en metadata: 2867

Ejemplo barcodes AnnData:
['AAACCTGAGGCGTACA-1-0', 'AAACGGGAGATCGATA-1-0', 'AAAGCAAAGATCGGGT-1-0', 'AAATGCCGTCTCAACA-1-0', 'AACACGTCACGGCTAC-1-0']

Ejemplo barcodes metadata HCC:
['AAACCTGAGGCGTACA-1', 'AAACGGGAGATCGATA-1', 'AAAGCAAAGATCGGGT-1', 'AAATGCCGTCTCAACA-1', 'AACACGTCACGGCTAC-1']


In [16]:
# Normalizar barcodes del AnnData quitando el último sufijo (-0 o -1)
adata.obs["barcode_clean"] = adata.obs_names.str.rsplit("-", n=1).str[0]

# Filtrar células HCC
hcc_barcodes_set = set(hcc_barcodes)
mask = adata.obs["barcode_clean"].isin(hcc_barcodes_set)

print(f"Células totales en AnnData: {adata.n_obs}")
print(f"Células HCC identificadas:  {mask.sum()}")
print(f"Células iCCA a eliminar:    {(~mask).sum()}")

# Filtrar
adata_hcc = adata[mask].copy()
print(f"\nAnnData solo HCC: {adata_hcc.shape}")

Células totales en AnnData: 9946
Células HCC identificadas:  2867
Células iCCA a eliminar:    7079

AnnData solo HCC: (2867, 21324)


In [17]:
# Guardar el AnnData filtrado solo con HCC
out_path = "/beegfs/home/iruizdealda/HCC_singlecell_project/data/adata_HCC_iCCA_raw.h5ad"
adata_hcc.write(out_path)
print(f"Guardado en: {out_path}")
print(f"Shape final: {adata_hcc.shape}")

Guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/data/adata_HCC_iCCA_raw.h5ad
Shape final: (2867, 21324)
